# 05 Data-Centre Metric Review and RA Workbook

Purpose: reduce the broad data-centre candidate metric extraction into a focused, auditable review workbook before comparative synthesis.

This notebook does **not** make final evidence claims. It creates a human-review workbook that separates candidate evidence from verified evidence.


In [ ]:
from pathlib import Path
import re
import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'Research_Workflow' / '02_Source_Extraction' / 'data' / 'coding_outputs' / 'candidate_metric_extractions.csv').exists():
            return p
        nested = p / 'DC_job_potential'
        if (nested / 'Research_Workflow' / '02_Source_Extraction' / 'data' / 'coding_outputs' / 'candidate_metric_extractions.csv').exists():
            return nested
    raise FileNotFoundError('Could not locate DC_job_potential project root from current working directory.')

PROJECT_ROOT = find_project_root()
BASE = PROJECT_ROOT / 'Research_Workflow' / '02_Source_Extraction'
INPUT_DIR = BASE / 'data' / 'coding_outputs'
OUTPUT_DIR = INPUT_DIR / 'data_centre_review' / 'metric_extraction'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_CSV = INPUT_DIR / 'candidate_metric_extractions.csv'
FULLTEXT_INVENTORY_CSV = INPUT_DIR / 'fulltext_inventory.csv'
SOURCE_STATUS_CSV = INPUT_DIR / 'source_recovery_status.csv'

print('Project root:', PROJECT_ROOT)
print('Output dir:', OUTPUT_DIR)


In [ ]:
candidates = pd.read_csv(CANDIDATES_CSV)
fulltext_inventory = pd.read_csv(FULLTEXT_INVENTORY_CSV)
source_status = pd.read_csv(SOURCE_STATUS_CSV)

print('candidate rows:', len(candidates))
print('sources:', candidates['source_id'].nunique())
display(candidates['metric'].value_counts().to_frame('rows'))
display(candidates['source_id'].value_counts().head(25).to_frame('rows'))


In [ ]:
def normalise_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

metric_weight = {
    'jobs_per_mw': 11,
    'operational_jobs_fte': 10,
    'construction_jobs_peak': 10,
    'construction_duration_months': 6,
    'construction_duration_years': 6,
    'capex': 5,
    'capacity_mw': 4,
    'gfa_sqft': 3,
    'causal_percent_effect': 5,
}

phase_weight = {
    'construction': 4,
    'operations': 3,
    'lifecycle_or_economy': 3,
    'unspecified': 0,
}

source_group_weight = {
    'causal_working_paper': 5,
    'causal_summary': 4,
    'government_audit': 4,
    'academic_labour': 4,
    'academic_review': 3,
    'australia_statistics': 4,
    'australia_skills': 4,
    'australia_industry': 3,
    'economic_impact_report': 3,
    'industry_io': 2,
    'industry_case': 2,
    'workforce_forecast': 3,
    'market_capex': 2,
    'market_context': 1,
    'advocacy_critique': 2,
    'subsidy_transparency': 2,
    'incentives_working_paper': 3,
    'regional_policy': 2,
}

positive_patterns = [
    r'\bjob', r'employ', r'workforce', r'worker', r'fte', r'full[- ]time',
    r'construction', r'operation', r'operational', r'direct', r'indirect', r'induced',
    r'capex', r'capital', r'invest', r'\$|usd|aud|million|billion',
    r'mw|megawatt|capacity', r'local econom', r'county', r'region', r'spillover',
]
negative_patterns = [
    r'table of contents', r'copyright', r'all rights reserved', r'disclaimer', r'appendix title',
]

def pattern_score(text, patterns, points):
    text_l = text.lower()
    return sum(points for pat in patterns if re.search(pat, text_l))

scored = candidates.copy()
scored['context_quote_norm'] = scored['context_quote'].map(normalise_text)
scored['quote_short'] = scored['context_quote_norm'].str.slice(0, 220)
scored['priority_score'] = (
    scored['metric'].map(metric_weight).fillna(0)
    + scored['phase_guess'].map(phase_weight).fillna(0)
    + scored['source_group'].map(source_group_weight).fillna(0)
)
scored['priority_score'] += scored['context_quote_norm'].map(lambda x: pattern_score(x, positive_patterns, 1))
scored['priority_score'] -= scored['context_quote_norm'].map(lambda x: pattern_score(x, negative_patterns, 4))
scored.loc[scored['source_id'].eq('dc_pwc_dcc_2025'), 'priority_score'] -= 2  # avoid letting one long report dominate the review workbook
scored.loc[scored['metric'].isin(['operational_jobs_fte', 'construction_jobs_peak', 'jobs_per_mw']), 'priority_score'] += 4
scored['priority_score'] = scored['priority_score'].clip(lower=0)

dedupe_cols = ['source_id', 'metric', 'raw_value', 'numeric_value', 'page_or_location', 'quote_short']
deduped = scored.sort_values('priority_score', ascending=False).drop_duplicates(dedupe_cols).copy()

print('deduped rows:', len(deduped), 'from', len(scored))
display(deduped[['source_id','metric','raw_value','phase_guess','priority_score','context_quote']].head(15))


In [ ]:
# Build a focused priority set. The intention is to catch the best evidence without burying the reviewer in thousands of table cells.
job_metrics = ['jobs_per_mw', 'operational_jobs_fte', 'construction_jobs_peak']

# Source caps keep large reports from dominating. These are review-workbook caps, not evidence judgments.
source_caps = {
    'dc_hamm_2025': 18,
    'dc_pwc_dcc_2025': 12,
    'dc_pham_2017': 8,
    'dc_yue_zeng_2026': 8,
    'dc_food_water_watch_2026': 8,
    'dc_alvarez_nber_2026': 8,
    'dc_nsw_skills_submission_2026': 6,
}
default_source_cap = 4

priority_parts = []

# Direct job metrics are central, but cap per source/metric to reduce repeated table fragments.
direct_jobs = (
    deduped[deduped['metric'].isin(job_metrics)]
    .sort_values('priority_score', ascending=False)
    .groupby(['source_id', 'metric'], group_keys=False)
    .head(5)
)
priority_parts.append(direct_jobs)

# Represent each source so the reviewer can decide whether it has any useful evidence.
priority_parts.append(
    deduped.sort_values('priority_score', ascending=False)
    .groupby('source_id', group_keys=False)
    .head(4)
)

# Represent each metric family, especially capex/capacity/duration context needed for synthesis.
priority_parts.append(
    deduped.sort_values('priority_score', ascending=False)
    .groupby('metric', group_keys=False)
    .head(12)
)

priority_pool = pd.concat(priority_parts, ignore_index=True).drop_duplicates(dedupe_cols).copy()
priority_pool = priority_pool.sort_values(['priority_score', 'source_id', 'metric'], ascending=[False, True, True])

capped_parts = []
for source_id, group in priority_pool.groupby('source_id', sort=False):
    cap = source_caps.get(source_id, default_source_cap)
    capped_parts.append(group.head(cap))

priority = pd.concat(capped_parts, ignore_index=True).copy()
priority = priority.sort_values(['priority_score', 'source_id', 'metric'], ascending=[False, True, True]).head(90).copy()
priority.insert(0, 'review_id', [f'dc_metric_{i:04d}' for i in range(1, len(priority) + 1)])

source_path_map = fulltext_inventory.set_index('source_id')['source_raw_path'].to_dict()
priority['source_raw_path'] = priority['source_id'].map(source_path_map)
priority['ra_decision'] = 'not_started'
priority['verified_value'] = ''
priority['verified_metric_definition'] = ''
priority['direct_vs_total'] = ''
priority['projected_vs_realised'] = ''
priority['geography'] = ''
priority['currency_price_year'] = ''
priority['notes'] = ''

def reason(row):
    bits = []
    if row['metric'] in job_metrics:
        bits.append('direct job metric')
    if row['metric'] in ['capex', 'capacity_mw']:
        bits.append('scale/investment context')
    if row['source_group'] in ['causal_working_paper', 'causal_summary']:
        bits.append('causal employment source')
    if row['source_group'] in ['australia_statistics', 'australia_skills', 'australia_industry']:
        bits.append('Australian context')
    if row['source_group'] in ['government_audit', 'subsidy_transparency', 'advocacy_critique']:
        bits.append('policy/critical evidence')
    return '; '.join(bits) or 'high scoring candidate'

priority['priority_reason'] = priority.apply(reason, axis=1)

review_cols = [
    'review_id','source_id','asset_type','source_group','title','metric','raw_value','numeric_value',
    'phase_guess','page_or_location','priority_score','priority_reason','context_quote','text_file','source_raw_path',
    'verification_status','ra_decision','verified_value','verified_metric_definition','direct_vs_total',
    'projected_vs_realised','geography','currency_price_year','notes'
]
priority_review = priority[[c for c in review_cols if c in priority.columns]].copy()

print('priority review rows:', len(priority_review))
display(priority_review['metric'].value_counts().to_frame('priority_rows'))
display(priority_review['source_id'].value_counts().to_frame('priority_rows'))


In [ ]:
source_summary = (
    candidates.groupby(['source_id','source_group','title'], dropna=False)
    .agg(candidate_rows=('source_id','size'), metrics=('metric', lambda s: ', '.join(sorted(s.dropna().unique()))))
    .reset_index()
)
source_summary['priority_review_rows'] = source_summary['source_id'].map(priority_review['source_id'].value_counts()).fillna(0).astype(int)
source_summary = source_summary.sort_values(['priority_review_rows','candidate_rows'], ascending=False)

all_scored_cols = [
    'source_id','asset_type','source_group','title','metric','raw_value','numeric_value','phase_guess',
    'page_or_location','priority_score','context_quote','text_file','verification_status'
]
all_candidates_scored = scored[[c for c in all_scored_cols if c in scored.columns]].sort_values('priority_score', ascending=False)

readme = pd.DataFrame({
    'field': ['purpose','review_decisions','important_warning','next_step'],
    'value': [
        'Focused review workbook generated from data-centre candidate metric extractions.',
        'Use ra_decision values such as accept, accept_scale, partial_accept, duplicate, reject, context_only, flag_for_verification.',
        'Rows are machine-prioritised candidates. Do not cite them until reviewed against the source text/PDF.',
        'After review, save a copy with a reviewed filename, then run the later synthesis notebook.'
    ]
})

priority_csv = OUTPUT_DIR / 'data_centre_priority_metric_review.csv'
all_scored_csv = OUTPUT_DIR / 'data_centre_candidate_metrics_scored.csv'
summary_csv = OUTPUT_DIR / 'data_centre_metric_source_summary.csv'
workbook_xlsx = OUTPUT_DIR / 'data_centre_metric_review_workbook.xlsx'

priority_review.to_csv(priority_csv, index=False)
all_candidates_scored.to_csv(all_scored_csv, index=False)
source_summary.to_csv(summary_csv, index=False)

with pd.ExcelWriter(workbook_xlsx, engine='openpyxl') as writer:
    readme.to_excel(writer, sheet_name='README', index=False)
    source_summary.to_excel(writer, sheet_name='source_summary', index=False)
    priority_review.to_excel(writer, sheet_name='priority_metric_review', index=False)
    all_candidates_scored.head(1500).to_excel(writer, sheet_name='all_candidate_metrics_top1500', index=False)
    fulltext_inventory.to_excel(writer, sheet_name='fulltext_inventory', index=False)

print('Wrote:', priority_csv)
print('Wrote:', all_scored_csv)
print('Wrote:', summary_csv)
print('Wrote:', workbook_xlsx)


In [ ]:
summary_md = OUTPUT_DIR / 'DATA_CENTRE_REVIEW_PREP_SUMMARY.md'
decision_text = f'''# Data-Centre Metric Review Prep Summary

Date: 2026-06-22

## Inputs

- Candidate metric rows: {len(candidates):,}
- Deduped candidate rows: {len(deduped):,}
- Source count: {candidates['source_id'].nunique():,}

## Outputs

- `data_centre_metric_review_workbook.xlsx`
- `data_centre_priority_metric_review.csv`
- `data_centre_candidate_metrics_scored.csv`
- `data_centre_metric_source_summary.csv`

## Review Workbook

The priority review sheet contains {len(priority_review):,} rows. These rows are still unverified and require human review before synthesis.

Recommended `ra_decision` values: `accept`, `accept_scale`, `partial_accept`, `duplicate`, `reject`, `context_only`, `flag_for_verification`.

## Important Caution

The PwC/DCC economic contribution report dominates the raw extraction because it contains many large tables. The notebook intentionally caps and scores candidates so the review workbook does not over-weight that one source.

The final comparative synthesis should only use rows that have been reviewed in this workbook or in a reviewed copy of it.
'''
summary_md.write_text(decision_text, encoding='utf-8')
print(summary_md)
print(decision_text)
